# 기초 B5 · 결측

이 노트북은 구글 **Colab**에서 바로 실행됩니다. 설치는 없고, 구글 계정만 있으면 됩니다.

**읽는 법.** 지금 읽고 있는 이런 글은 설명이라 실행할 것이 없습니다. 코드는 **왼쪽에 `[ ]` 와 ▶ 단추가 붙은 상자**이고, 그 ▶ 또는 `Shift`+`Enter` 로 실행합니다.

**순서.** 번호 차례대로 끝까지 갑니다. **1 준비**부터 시작해 마지막 번호까지 위에서 아래로 내려가면 됩니다. ⚠ 가운데부터 누르면 앞에서 만든 것이 없어 오류가 납니다.

📖 본문 학습 페이지: [기초 B5 · 결측](https://grow.minds.kr/textbooks/css-methods/causal/book/b5-결측.html)

## 1. 준비

아래 상자 **둘**을 차례로 실행하세요. 둘 다 해야 그다음이 돌아갑니다.

1. 첫째 = 자료와 코드를 내려받습니다. 몇 초 걸리고 「경고」 문구가 떠도 정상입니다. 상자 아래에 `/content/css-methods-causal-code` 가 찍히면 성공입니다.
2. 둘째 = 도구를 불러옵니다. **준비 끝**이라는 줄이 찍히면 됩니다.

⚠ 한참 뒤에 돌아와 `pd`·`exp`가 없다거나 파일이 없다는 오류가 나면 코랩이 연결을 끊은 것입니다. 자료는 내 컴퓨터가 아니라 코랩의 컴퓨터에 있었으므로 함께 사라졌습니다. **이 두 상자를 위에서부터 다시** 누르면 됩니다(첫째 상자는 여러 번 눌러도 됩니다).

> **어느 것이 상자인가.** 왼쪽에 `[ ]` 표시와 **▶** 단추가 붙은 것이 코드 상자입니다. 글씨에 색이 섞여 있습니다. 지금 읽고 있는 이런 글에는 `[ ]` 도 ▶ 도 없습니다.

In [ ]:
# 이 책의 데이터·코드를 코랩으로 내려받습니다(수 초). 다시 눌러도 됩니다.
%cd /content
![ -d css-methods-causal-code ] || git clone -q https://github.com/dataminds/css-methods-causal-code.git
%cd /content/css-methods-causal-code

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

def load(name, clean=True):
    df = pd.read_csv(f"data/journey_{name}.csv")
    return df[df.attn_1 == 1] if clean and "attn_1" in df else df

def ols(y, X):                      # 절편 포함 최소제곱 → (계수, 표준오차, p, R^2)
    y = np.asarray(y, float)
    X1 = np.column_stack([np.ones(len(y))] + [np.asarray(x, float) for x in X])
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ b
    n, k = X1.shape
    se = np.sqrt(np.diag(resid @ resid / (n - k) * np.linalg.inv(X1.T @ X1)))
    p = 2 * stats.t.sf(np.abs(b / se), n - k)
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
    return b, se, p, r2

def cohen_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

def cronbach(items):
    items = np.asarray(items, float); k = items.shape[1]
    return k/(k-1) * (1 - items.var(axis=0, ddof=1).sum() / items.sum(axis=1).var(ddof=1))

print("준비 끝. 데이터와 도우미 함수를 불러왔습니다.")


💡 **방금 둘째 상자에 코드가 길게 펼쳐졌는데, 지금 이해하지 않아도 됩니다.**

이 책 전체가 쓰는 **도우미 넷**을 미리 만들어 두는 곳이라 그렇습니다. 지금은 이름과 하는 일만 훑고 넘어가세요. 안에 든 식은 필요한 장에서 하나씩 만납니다.

| 이름 | 하는 일 | 처음 쓰는 곳 |
|---|---|---|
| `load` | 자료 한 벌을 불러오고 주의 점검 실패자를 걸러 낸다 | 2장 |
| `cronbach` | 척도 신뢰도(알파) | 4장 |
| `cohen_d` | 두 집단 차이의 효과크기 | 13장 |
| `ols` | 회귀의 계수·표준오차·p·R² | 14장 |

⚠ `np.linalg.lstsq` 처럼 낯선 이름이 보여도 괜찮습니다. **그것이 이 책이 「계산은 AI가」라고 말하는 곳입니다.**

⭐ **그리고 이 넷은 여러분이 검증할 대상이 아닙니다.** 이 책이 검증해 둔 도구입니다(배포 때마다 대조 배터리가 이 함수들의 출력을 본문 수치와 맞춰 봅니다). 여러분이 읽어야 할 코드는 **AI가 준 코드**이고, 그 훈련은 3장부터 시작합니다.

📖 이 넷을 다시 보고 싶으면 책 부록 E.1b 에 같은 코드가 있습니다.

## 2. (책이 원리를 보이려고 돌린 코드) 참값과 방법별 추정 (책 B5.3)
패널판은 모의 세계라서 생성기가 빈칸을 만들기 전의 값(`data/truth_panel.csv`)을 다시 만들어 두었습니다. 그 참값과, 빈칸이 있는 자료로 여러 방법을 써서 구한 3차 평균 · 1차–3차 상관 · 기울기를 견줍니다. 완전정보 최우추정은 R 의 lavaan 대신 같은 추정을 내는 EM 알고리즘으로 계산합니다.

In [ ]:
def wide(df):
    m = df.pivot(index="id", columns="wave", values="mil")
    h = df.pivot(index="id", columns="wave", values="hjs")
    return pd.DataFrame({"m1": m[1], "m2": m[2], "m3": m[3], "h1": h[1], "h2": h[2]})
obs, true = wide(load("panel")), wide(pd.read_csv("data/truth_panel.csv"))   # 참값은 분석 자료가 아니라 이름이 다르다
def three(d):
    ok = d[["m1", "m3"]].dropna()
    b = np.cov(ok.m1, ok.m3)[0, 1] / ok.m1.var()
    return [float(round(v, 3)) for v in (d.m3.mean(), ok.m1.corr(ok.m3), b)]
print("참값      ", three(true))
print("목록별 삭제", three(obs.dropna(subset=["m3"])))
print("평균 대치  ", three(obs.assign(m3=obs.m3.fillna(obs.m3.mean()))))
r = obs.copy()
for y, xs in [("m2", ["m1", "h1"]), ("m3", ["m1", "m2", "h1"])]:   # 한 번 회귀 대치
    ok = obs[y].notna()
    bb, *_ = ols(r.loc[ok, y], [r.loc[ok, x] for x in xs])
    r.loc[~ok, y] = bb[0] + sum(bb[i + 1] * r.loc[~ok, x] for i, x in enumerate(xs))
print("회귀 대치  ", three(r))
Y = obs[["m1", "m2", "m3", "h1", "h2"]].values                    # EM = 완전정보 최우추정의 점추정
mu, S = np.nanmean(Y, 0), np.diag(np.nanvar(Y, 0))
for _ in range(2000):
    T1, T2 = np.zeros(5), np.zeros((5, 5))
    for row in Y:
        o = ~np.isnan(row); x = row.copy(); C = np.zeros((5, 5))
        if (~o).any():
            K = S[np.ix_(~o, o)] @ np.linalg.inv(S[np.ix_(o, o)])
            x[~o] = mu[~o] + K @ (row[o] - mu[o])
            C[np.ix_(~o, ~o)] = S[np.ix_(~o, ~o)] - K @ S[np.ix_(~o, o)].T
        T1 += x; T2 += np.outer(x, x) + C
    mu2 = T1 / len(Y); S2 = T2 / len(Y) - np.outer(mu2, mu2)
    done = np.abs(mu2 - mu).max() < 1e-13 and np.abs(S2 - S).max() < 1e-13
    mu, S = mu2, S2
    if done:
        break
print("완전정보 최우추정", [float(round(v, 3)) for v in (mu[2], S[0, 2] / np.sqrt(S[0, 0] * S[2, 2]), S[0, 2] / S[0, 0])])
# 참값       [4.774, 0.456, 0.481]
# 목록별 삭제 [4.926, 0.453, 0.493]
# 평균 대치   [4.926, 0.363, 0.317]
# 회귀 대치   [4.78, 0.538, 0.517]
# 완전정보 최우추정 [4.78, 0.486, 0.518]

## 3. (책이 원리를 보이려고 돌린 코드) 다중 대치는 대치할 때마다 얼마나 흔들리나 (책 B5.4 상자)
책 「AI가 준 코드 읽기」 코드 2 의 다중 대치를 함수로 만들어, 씨앗만 1 에서 100 까지 바꿔 돌립니다. 3차 평균 추정이 대치할 때마다 얼마나 흔들리는지(표준편차) 셉니다. 1분쯤 걸립니다.

In [ ]:
def mi(seed, m=20):
    g = np.random.default_rng(seed); z0 = obs[["m1", "m2", "m3", "h1"]]; out = []
    for _ in range(m):
        z = z0.copy()
        for y, xs in [("m2", ["m1", "h1"]), ("m3", ["m1", "m2", "h1"])]:
            ok = z0[y].notna()
            X = np.column_stack([np.ones(ok.sum())] + [z.loc[ok, x] for x in xs])
            b = np.linalg.lstsq(X, z.loc[ok, y], rcond=None)[0]; e = z.loc[ok, y] - X @ b
            s2 = (e @ e) / g.chisquare(len(e) - len(b))
            bd = g.multivariate_normal(b, s2 * np.linalg.inv(X.T @ X))
            Xm = np.column_stack([np.ones((~ok).sum())] + [z.loc[~ok, x] for x in xs])
            z.loc[~ok, y] = Xm @ bd + g.normal(0, np.sqrt(s2), (~ok).sum())
        out.append(z.m3.mean())
    return np.mean(out)
print(round(np.std([mi(s) for s in range(1, 101)], ddof=1), 4))
# 0.0066

## 4. 직접 바꿔 보기
위 셀의 숫자(씨앗 73, 표본 크기, 제외 기준 등)를 바꿔 다시 실행해 보세요. 결과가 어떻게 달라지나요?

> **검증 로그(부록 B)**: 무엇을 바꿨고, 무엇이 나왔고, 예상과 같았는지 한 문단으로 적어 두세요. 실행이 아니라 검증이 이 책의 핵심입니다.